# **US Electric Power Sector CO₂ Emission Forecasting**
### **Mathematics for Computing - 2 ()**
---
This notebook delivers a production-grade machine learning and deep learning forecasting framework for monthly carbon dioxide emissions from the US electric power sector.

### **Pipeline Highlights:**
1. **Advanced Data Preprocessing**:
   - Clean column standardization and chronological sequencing (1973–2016).
   - Domain-aware missing value resolution (pre-monitoring zero baselines for Geothermal and Non-Biomass Waste + linear interpolation).
   - Outlier detection and soft winsorization (1st and 99th percentile IQR capping).
2. **Time-Series Feature Engineering**:
   - Harmonic/cyclical temporal encoding ($Month_{sin}, Month_{cos}$).
   - Multi-scale autoregressive lags ($t-1, t-2, t-3, t-12$ annual seasonal cycle).
   - Rolling statistics (3, 6, and 12-month rolling means, rolling volatility, and EMAs).
   - Dynamic fuel market shares ($Coal_{share}, Gas_{share}, Petroleum_{share}$).
3. **Leakage-Free Validation & Scaling**:
   - Strict chronological train-test split (85% train, 15% holdout test).
   - 5-fold `TimeSeriesSplit` cross-validation without lookahead bias.
4. **Hyperparameter Tuning**:
   - Systematic `GridSearchCV` on Ridge Regression, LightGBM, XGBoost, and SVR.
5. **Deep Sequential Architectures**:
   - Multi-feature Bidirectional LSTM and Stacked GRU with Dropout and EarlyStopping.
6. **Interpretability & Benchmarking**:
   - Comprehensive metric comparison, feature importances, and residual error distributions.


## **1. Data Preprocessing & Advanced Feature Engineering**


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

# Load dataset (checking relative and workspace paths)
data_file = 'cleaned_monthly_sectoral_dataset.csv'
if not os.path.exists(data_file):
    data_file = os.path.join('UI', 'cleaned_monthly_sectoral_dataset.csv')

df = pd.read_csv(data_file)
print(f"Loaded raw dataset with shape: {df.shape}")

# Standardize column names
column_mapping = {
    'Date': 'Date',
    'Coal Electric Power Sector CO2 Emissions': 'Coal',
    'Distillate Fuel, Including Kerosene-Type Jet Fuel, Oil Electric Power Sector CO2 Emissions': 'Distillate_Fuel',
    'Geothermal Energy Electric Power Sector CO2 Emissions': 'Geothermal',
    'Natural Gas Electric Power Sector CO2 Emissions': 'Natural_Gas',
    'Non-Biomass Waste Electric Power Sector CO2 Emissions': 'Non_Biomass_Waste',
    'Petroleum Coke Electric Power Sector CO2 Emissions': 'Petroleum_Coke',
    'Petroleum Electric Power Sector CO2 Emissions': 'Petroleum',
    'Residual Fuel Oil Electric Power Sector CO2 Emissions': 'Residual_Fuel_Oil',
    'Total Energy Electric Power Sector CO2 Emissions': 'Total_CO2'
}
df.rename(columns=column_mapping, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

# Domain-aware missing value treatment
for col in ['Geothermal', 'Non_Biomass_Waste']:
    first_valid = df[col].first_valid_index()
    if first_valid is not None and first_valid > 0:
        df.loc[:first_valid-1, col] = 0.0

df.interpolate(method='linear', inplace=True)
df.ffill(inplace=True)
df.bfill(inplace=True)

# Outlier Winsorization (1st and 99th percentiles)
feature_cols_raw = [c for c in df.columns if c not in ['Date', 'Total_CO2']]
for col in feature_cols_raw:
    q1 = df[col].quantile(0.01)
    q3 = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=q1, upper=q3)

# Temporal & Cyclical Features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['DayOfYear'] = df['Date'].dt.dayofyear
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12.0)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12.0)

# Autoregressive Lags
df['Lag_1'] = df['Total_CO2'].shift(1)
df['Lag_2'] = df['Total_CO2'].shift(2)
df['Lag_3'] = df['Total_CO2'].shift(3)
df['Lag_12'] = df['Total_CO2'].shift(12)  # Annual seasonal cycle
df['Coal_Lag1'] = df['Coal'].shift(1)
df['Natural_Gas_Lag1'] = df['Natural_Gas'].shift(1)
df['Petroleum_Lag1'] = df['Petroleum'].shift(1)

# Rolling Window Volatility & Smoothing
df['Roll_Mean_3'] = df['Total_CO2'].shift(1).rolling(window=3).mean()
df['Roll_Mean_6'] = df['Total_CO2'].shift(1).rolling(window=6).mean()
df['Roll_Mean_12'] = df['Total_CO2'].shift(1).rolling(window=12).mean()
df['Roll_Std_3'] = df['Total_CO2'].shift(1).rolling(window=3).std().fillna(0)
df['Roll_Std_6'] = df['Total_CO2'].shift(1).rolling(window=6).std().fillna(0)
df['EMA_3'] = df['Total_CO2'].shift(1).ewm(span=3, adjust=False).mean()
df['EMA_6'] = df['Total_CO2'].shift(1).ewm(span=6, adjust=False).mean()

# Fuel Shares & Differences
df['Coal_Share'] = df['Coal'] / (df['Total_CO2'] + 1e-5)
df['Natural_Gas_Share'] = df['Natural_Gas'] / (df['Total_CO2'] + 1e-5)
df['Petroleum_Share'] = df['Petroleum'] / (df['Total_CO2'] + 1e-5)
df['Diff_1'] = df['Total_CO2'].shift(1) - df['Total_CO2'].shift(2)
df['Diff_12'] = df['Total_CO2'].shift(1) - df['Total_CO2'].shift(13)

df_clean = df.dropna().reset_index(drop=True)
print(f"Preprocessed feature matrix shape: {df_clean.shape}")

# Chronological Train-Test Split (85% Train, 15% Holdout Test)
target_col = 'Total_CO2'
ignore_cols = ['Date', target_col]
feature_names = [c for c in df_clean.columns if c not in ignore_cols]

split_idx = int(len(df_clean) * 0.85)
train_df = df_clean.iloc[:split_idx]
test_df = df_clean.iloc[split_idx:]

X_train_raw = train_df[feature_names].values
y_train_raw = train_df[target_col].values.reshape(-1, 1)

X_test_raw = test_df[feature_names].values
y_test_raw = test_df[target_col].values.reshape(-1, 1)

# Scalers fit strictly on train set
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train_raw)
X_test = scaler_X.transform(X_test_raw)

scaler_y = MinMaxScaler(feature_range=(0, 1))
y_train = scaler_y.fit_transform(y_train_raw).ravel()
y_test = scaler_y.transform(y_test_raw).ravel()

tscv = TimeSeriesSplit(n_splits=5)
all_results = []
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)} | Features: {len(feature_names)}")


## **2. Regularized Linear Regression (Ridge) with Hyperparameter Tuning**


In [ ]:
print("Tuning Ridge Regression with TimeSeriesSplit...")
ridge_params = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0, 200.0],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}
ridge_cv = GridSearchCV(Ridge(random_state=42), ridge_params, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
ridge_cv.fit(X_train, y_train)
best_ridge = ridge_cv.best_estimator_

y_pred_ridge = best_ridge.predict(X_test)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mse_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

all_results.append({'Model': 'Ridge Regression', 'MSE': mse_ridge, 'RMSE': rmse_ridge, 'MAE': mae_ridge, 'R2': r2_ridge, 'Best_Params': str(ridge_cv.best_params_)})

print("Best Parameters:", ridge_cv.best_params_)
print(f"Ridge Regression Performance (Normalized): RMSE={rmse_ridge:.4f}, MAE={mae_ridge:.4f}, MSE={mse_ridge:.6f}, R2={r2_ridge:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(test_df['Date'], y_test, label='Actual (Normalized)', color='#1e293b', linewidth=2)
plt.plot(test_df['Date'], y_pred_ridge, label='Ridge Predicted', color='#3b82f6', linestyle='--', linewidth=2)
plt.title('Ridge Regression: Actual vs Predicted CO₂ Emissions', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Normalized Emissions [0, 1]')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


## **3. LightGBM Regressor with Hyperparameter Tuning**


In [ ]:
print("Tuning LightGBM Regressor with TimeSeriesSplit...")
lgb_param_grid = {
    'n_estimators': [150, 300],
    'learning_rate': [0.03, 0.08],
    'num_leaves': [15, 31],
    'max_depth': [4, -1]
}
lgb_base = LGBMRegressor(random_state=42, verbose=-1)
lgb_cv = GridSearchCV(lgb_base, lgb_param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
lgb_cv.fit(X_train, y_train)
best_lgb = lgb_cv.best_estimator_

y_pred_lgb = best_lgb.predict(X_test)
mse_lgb = mean_squared_error(y_test, y_pred_lgb)
rmse_lgb = np.sqrt(mse_lgb)
mae_lgb = mean_absolute_error(y_test, y_pred_lgb)
r2_lgb = r2_score(y_test, y_pred_lgb)

all_results.append({'Model': 'LightGBM', 'MSE': mse_lgb, 'RMSE': rmse_lgb, 'MAE': mae_lgb, 'R2': r2_lgb, 'Best_Params': str(lgb_cv.best_params_)})

print("Best Parameters:", lgb_cv.best_params_)
print(f"LightGBM Performance (Normalized): RMSE={rmse_lgb:.4f}, MAE={mae_lgb:.4f}, MSE={mse_lgb:.6f}, R2={r2_lgb:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(test_df['Date'], y_test, label='Actual (Normalized)', color='#1e293b', linewidth=2)
plt.plot(test_df['Date'], y_pred_lgb, label='LightGBM Predicted', color='#10b981', linestyle='--', linewidth=2)
plt.title('LightGBM: Actual vs Predicted CO₂ Emissions', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Normalized Emissions [0, 1]')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


## **4. XGBoost Regressor with Hyperparameter Tuning**


In [ ]:
print("Tuning XGBoost Regressor with TimeSeriesSplit...")
xgb_param_grid = {
    'n_estimators': [150, 300],
    'learning_rate': [0.03, 0.08],
    'max_depth': [3, 5]
}
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')
xgb_cv = GridSearchCV(xgb_base, xgb_param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
xgb_cv.fit(X_train, y_train)
best_xgb = xgb_cv.best_estimator_

y_pred_xgb = best_xgb.predict(X_test)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

all_results.append({'Model': 'XGBoost', 'MSE': mse_xgb, 'RMSE': rmse_xgb, 'MAE': mae_xgb, 'R2': r2_xgb, 'Best_Params': str(xgb_cv.best_params_)})

print("Best Parameters:", xgb_cv.best_params_)
print(f"XGBoost Performance (Normalized): RMSE={rmse_xgb:.4f}, MAE={mae_xgb:.4f}, MSE={mse_xgb:.6f}, R2={r2_xgb:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(test_df['Date'], y_test, label='Actual (Normalized)', color='#1e293b', linewidth=2)
plt.plot(test_df['Date'], y_pred_xgb, label='XGBoost Predicted', color='#f59e0b', linestyle='--', linewidth=2)
plt.title('XGBoost: Actual vs Predicted CO₂ Emissions', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Normalized Emissions [0, 1]')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


## **5. Support Vector Regressor (SVR) with Hyperparameter Tuning**


In [ ]:
print("Tuning Support Vector Regressor (SVR)...")
svr_param_grid = {
    'C': [1.0, 10.0, 50.0],
    'gamma': ['scale', 0.05],
    'epsilon': [0.01, 0.05],
    'kernel': ['rbf']
}
svr_base = SVR()
svr_cv = GridSearchCV(svr_base, svr_param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
svr_cv.fit(X_train, y_train)
best_svr = svr_cv.best_estimator_

y_pred_svr = best_svr.predict(X_test)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)
r2_svr = r2_score(y_test, y_pred_svr)

all_results.append({'Model': 'SVM', 'MSE': mse_svr, 'RMSE': rmse_svr, 'MAE': mae_svr, 'R2': r2_svr, 'Best_Params': str(svr_cv.best_params_)})

print("Best Parameters:", svr_cv.best_params_)
print(f"SVR Performance (Normalized): RMSE={rmse_svr:.4f}, MAE={mae_svr:.4f}, MSE={mse_svr:.6f}, R2={r2_svr:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(test_df['Date'], y_test, label='Actual (Normalized)', color='#1e293b', linewidth=2)
plt.plot(test_df['Date'], y_pred_svr, label='SVM Predicted', color='#8b5cf6', linestyle='--', linewidth=2)
plt.title('Support Vector Regressor: Actual vs Predicted CO₂ Emissions', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Normalized Emissions [0, 1]')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


## **6. Deep Sequential Modeling: Bidirectional LSTM & GRU**


In [ ]:
print("Configuring Deep Learning Sequence Tensors (Lookback Window = 12)...")
lookback = 12

def create_dl_sequences(X_data, y_data, window):
    X_seq, y_seq = [], []
    for i in range(window, len(X_data)):
        X_seq.append(X_data[i-window:i, :])
        y_seq.append(y_data[i])
    return np.array(X_seq), np.array(y_seq)

full_X = np.vstack([X_train, X_test])
full_y = np.concatenate([y_train, y_test])
X_dl, y_dl = create_dl_sequences(full_X, full_y, lookback)

train_dl_size = len(X_train) - lookback
X_train_dl, X_test_dl = X_dl[:train_dl_size], X_dl[train_dl_size:]
y_train_dl, y_test_dl = y_dl[:train_dl_size], y_dl[train_dl_size:]

# 1. Deep Bidirectional LSTM
lstm_model = Sequential([
    Input(shape=(X_train_dl.shape[1], X_train_dl.shape[2])),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
callbacks = [
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-5, verbose=0)
]
history_lstm = lstm_model.fit(X_train_dl, y_train_dl, validation_data=(X_test_dl, y_test_dl), epochs=50, batch_size=16, callbacks=callbacks, verbose=0)
y_pred_lstm = lstm_model.predict(X_test_dl).flatten()
mse_lstm = mean_squared_error(y_test_dl, y_pred_lstm)
all_results.append({'Model': 'LSTM', 'MSE': mse_lstm, 'RMSE': np.sqrt(mse_lstm), 'MAE': mean_absolute_error(y_test_dl, y_pred_lstm), 'R2': r2_score(y_test_dl, y_pred_lstm), 'Best_Params': 'BiLSTM(64) + LSTM(32) + Dropout(0.2)'})

# 2. Gated Recurrent Unit (GRU)
gru_model = Sequential([
    Input(shape=(X_train_dl.shape[1], X_train_dl.shape[2])),
    GRU(64, return_sequences=True),
    Dropout(0.2),
    GRU(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
gru_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
history_gru = gru_model.fit(X_train_dl, y_train_dl, validation_data=(X_test_dl, y_test_dl), epochs=50, batch_size=16, callbacks=callbacks, verbose=0)
y_pred_gru = gru_model.predict(X_test_dl).flatten()
mse_gru = mean_squared_error(y_test_dl, y_pred_gru)
all_results.append({'Model': 'GRU', 'MSE': mse_gru, 'RMSE': np.sqrt(mse_gru), 'MAE': mean_absolute_error(y_test_dl, y_pred_gru), 'R2': r2_score(y_test_dl, y_pred_gru), 'Best_Params': 'Stacked GRU(64) + GRU(32) + Dropout(0.2)'})

print(f"Bidirectional LSTM R2: {r2_score(y_test_dl, y_pred_lstm):.4f} | RMSE: {np.sqrt(mse_lstm):.4f}")
print(f"Stacked GRU R2: {r2_score(y_test_dl, y_pred_gru):.4f} | RMSE: {np.sqrt(mse_gru):.4f}")

# Plot DL Forecasts & Loss Convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(test_df['Date'], y_test_dl, label='Actual', color='#1e293b', linewidth=2)
axes[0].plot(test_df['Date'], y_pred_lstm, label='BiLSTM', color='#0ea5e9', linestyle='--')
axes[0].plot(test_df['Date'], y_pred_gru, label='GRU', color='#ec4899', linestyle=':')
axes[0].set_title('Deep Learning Predictions vs Actual', fontweight='bold')
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].plot(history_lstm.history['val_loss'], label='LSTM Val Loss', color='#0ea5e9')
axes[1].plot(history_gru.history['val_loss'], label='GRU Val Loss', color='#ec4899')
axes[1].set_title('Validation Loss Convergence (MSE)', fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


## **7. Model Benchmarking Leaderboard & Sectoral Contribution**


In [ ]:
# Summary Results Table
results_df = pd.DataFrame(all_results).sort_values(by='R2', ascending=False).reset_index(drop=True)
print("=== COMPREHENSIVE MODEL BENCHMARKS ===")
print(results_df[['Model', 'R2', 'RMSE', 'MAE', 'MSE']])

# Bar Chart Comparison
plt.figure(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.2
plt.bar(x - 1.5*width, results_df['RMSE'], width, label='RMSE', color='#3b82f6')
plt.bar(x - 0.5*width, results_df['MAE'], width, label='MAE', color='#10b981')
plt.bar(x + 0.5*width, results_df['MSE'], width, label='MSE', color='#f59e0b')
plt.bar(x + 1.5*width, results_df['R2'], width, label='R²', color='#8b5cf6')
plt.xticks(x, results_df['Model'])
plt.ylabel('Metric Score')
plt.title('Model Benchmark Comparison Across Evaluation Metrics', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Sectoral Contribution Over Time
percent_df = df_clean.iloc[:, 1:9].div(df_clean['Total_CO2'], axis=0) * 100
percent_df['Date'] = df_clean['Date']

plt.figure(figsize=(14, 6))
for sector in df_clean.columns[1:9]:
    plt.plot(percent_df['Date'], percent_df[sector], label=sector)

plt.legend(title="Energy Sectors", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title("US Sector-wise CO₂ Contribution Over Time (%)", fontsize=13, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Contribution (%)")
plt.tight_layout()
plt.show()
